# Auto Coding Review

Review notebook for the first-pass machine-assisted coding draft. This notebook is read-only and is designed to highlight uncertainty, dataset coverage, and rows that should receive early human review.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("default")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [ ]:
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from config import COLLAPSE_DATASET_PATH
from dataset_utils import get_factor_columns, load_collapse_dataset

df = load_collapse_dataset(COLLAPSE_DATASET_PATH)
factor_cols = get_factor_columns()
factor_data = df[factor_cols].apply(pd.to_numeric, errors="coerce")

df.shape

## Factor Value Counts

These summaries show how often each score appears across the factor matrix.

In [ ]:
overall_value_counts = (
    factor_data.stack(future_stack=True)
    .value_counts(dropna=False)
    .rename_axis("factor_value")
    .to_frame("count")
    .sort_index()
)

factor_value_counts = pd.DataFrame(
    {
        score: (factor_data == score).sum()
        for score in [0, 1, 2, 3, 9]
    }
).sort_values(9, ascending=False)

display(overall_value_counts)
display(factor_value_counts)

fig, ax = plt.subplots(figsize=(7, 4))
overall_value_counts.loc[[0, 1, 2, 3, 9], "count"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Overall factor value counts")
ax.set_xlabel("Factor value")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Uncertainty By Factor and By Case

Score `9` is treated as structured uncertainty rather than missing data.

In [ ]:
unknown_9_by_factor = (factor_data == 9).sum().sort_values(ascending=False).to_frame("unknown_9_count")
unknown_9_by_case = (
    (factor_data == 9)
    .sum(axis=1)
    .groupby(df["case_name"])
    .agg(["mean", "max", "min"])
    .rename(columns={"mean": "avg_unknown_9", "max": "max_unknown_9", "min": "min_unknown_9"})
    .sort_values("avg_unknown_9", ascending=False)
)

display(unknown_9_by_factor)
display(unknown_9_by_case.round(2))

fig, ax = plt.subplots(figsize=(10, 10))
unknown_9_by_factor.sort_values("unknown_9_count").plot(kind="barh", ax=ax, legend=False, color="indianred")
ax.set_title("Count of 9 scores by factor")
ax.set_xlabel("Rows scored 9")
ax.set_ylabel("Factor")
plt.tight_layout()
plt.show()

## Coding Completeness By Case

Because the matrix is fully populated, this section measures known scores (`0` to `3`) rather than simple non-null completeness.

In [ ]:
known_scores_mask = factor_data.isin([0, 1, 2, 3])
coding_completeness_by_case = pd.DataFrame(
    {
        "rows": df.groupby("case_name").size(),
        "known_factor_scores": known_scores_mask.sum(axis=1).groupby(df["case_name"]).sum(),
        "unknown_9_scores": (factor_data == 9).sum(axis=1).groupby(df["case_name"]).sum(),
    }
)
coding_completeness_by_case["total_factor_slots"] = coding_completeness_by_case["rows"] * len(factor_cols)
coding_completeness_by_case["known_pct"] = (
    coding_completeness_by_case["known_factor_scores"] / coding_completeness_by_case["total_factor_slots"] * 100
).round(2)
coding_completeness_by_case = coding_completeness_by_case.sort_values("known_pct", ascending=False)

display(coding_completeness_by_case)

fig, ax = plt.subplots(figsize=(10, 7))
coding_completeness_by_case["known_pct"].sort_values().plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Known-score coverage by case")
ax.set_xlabel("Factor cells scored 0-3 (%)")
ax.set_ylabel("Case")
plt.tight_layout()
plt.show()

## Collapse Outcome Distribution By Case

In [ ]:
collapse_outcome_by_case = (
    df.pivot_table(index="case_name", columns="collapse_outcome", values="case_id", aggfunc="count", fill_value=0)
    .rename(columns={0: "stable", 1: "stressed", 2: "severe_decline", 3: "collapse"})
)

display(collapse_outcome_by_case)

fig, ax = plt.subplots(figsize=(12, 8))
collapse_outcome_by_case.plot(kind="bar", stacked=True, ax=ax, colormap="tab20c")
ax.set_title("Collapse outcome distribution by case")
ax.set_xlabel("Case")
ax.set_ylabel("Row count")
ax.legend(title="Outcome")
plt.tight_layout()
plt.show()

## Rows That Need Early Review

This section flags rows with especially high uncertainty and rows where collapse-like coding appears early within a case timeline.

In [ ]:
review_df = df[["case_id", "case_name", "period_start", "period_end", "phase_label", "collapse_outcome", "data_confidence", "notes"]].copy()
review_df["unknown_9_count"] = (factor_data == 9).sum(axis=1)
review_df["high_stress_factor_count"] = factor_data.ge(2).sum(axis=1)
review_df["known_factor_count"] = factor_data.isin([0, 1, 2, 3]).sum(axis=1)

case_order = df.groupby("case_name").cumcount() + 1
case_total_rows = df.groupby("case_name")["case_id"].transform("count")
review_df["row_position"] = case_order
review_df["case_total_rows"] = case_total_rows
review_df["relative_position"] = (review_df["row_position"] / review_df["case_total_rows"]).round(3)

nearly_all_9_rows = review_df[review_df["unknown_9_count"] >= 24].sort_values(["unknown_9_count", "case_name"], ascending=[False, True])
early_collapse_flags = review_df[
    ((review_df["relative_position"] <= 0.33) & (review_df["collapse_outcome"] >= 2))
    | ((review_df["relative_position"] <= 0.5) & (review_df["phase_label"] == "collapse"))
].sort_values(["case_name", "period_start"])

confidence_rank = review_df["data_confidence"].map({"low": 2, "medium": 1, "high": 0})
most_uncertain_rows = review_df.assign(confidence_rank=confidence_rank).sort_values(
    ["unknown_9_count", "confidence_rank", "relative_position"],
    ascending=[False, False, True],
).drop(columns="confidence_rank")

display(nearly_all_9_rows)
display(early_collapse_flags)
display(most_uncertain_rows.head(20))